# Thesis Phase 3 — Model Development & Comparison

**Author:** Ivan Tomilo (PJATK)  
**Thesis:** Comparing CNN, Vision Transformer, and Hybrid Architectures for Image Classification in Low-Data Regimes  
**Dataset:** CIFAR-10 (32×32 RGB, 10 classes)

## Phase 3 deliverables (per spec)

| Part | Content | Weight |
|---|---|---|
| A | **Advanced Model 1** — Vision Transformer (ViT-Small/4) | 10% |
| B | **Advanced Model 2** — Compact Convolutional Transformer (CCT-7/3×1) | 10% |
| C | **Final Model Comparison** — table, overfitting analysis, model selection | 10% |

### Rationale for model choices

- **Model 1: ViT-Small/4.** Pure transformer architecture. Tests the hypothesis that *without* convolutional inductive biases (locality, translation equivariance), a transformer trained from scratch struggles on a small dataset such as CIFAR-10. The `/4` denotes 4×4 patches, which yields 64 tokens on a 32×32 image — required because the canonical 16×16 patch size would yield only 4 tokens (Lee et al., 2021; Hassani et al., 2022).
- **Model 2: CCT-7/3×1.** Hybrid architecture from Hassani et al. (2022). Replaces patch tokenisation with a 3×3 convolutional tokenizer (introducing locality) and the `[CLS]` token with a learnable sequence-pooling layer. Reaches ~98 % on CIFAR-10 from scratch with only ~3.7 M parameters, making it specifically designed for the data-scarcity question this thesis investigates.

### Controlled experimental protocol (Part C requirement)

All three models share an **identical training recipe** — the only varying factor is the architecture. This is the core methodological requirement of the thesis.

| Element | Value |
|---|---|
| Optimiser | AdamW (β₁=0.9, β₂=0.999) |
| Base LR | 5e-4 |
| Weight decay | 0.05 |
| Schedule | Cosine annealing with linear warmup |
| Epochs | 50 (default for Colab budget; 100–200 recommended for final thesis numbers) |
| Warmup epochs | 5 |
| Batch size | 128 |
| Augmentation | RandAugment (N=2, M=9) + RandomCrop + HFlip + RandomErasing |
| Regularisation | Mixup (α=0.8), CutMix (α=1.0), stochastic depth, label smoothing 0.1 |
| Mixed precision | Yes (AMP) |
| Gradient clipping | 1.0 (global L2 norm) |
| Seed | 42 |
| Val split | Stratified 10 % held out from training data (consistent with Phase 2) |

> **Note on the Phase 2 baseline.** The Phase 2 ResNet-18 used a lighter augmentation recipe. To keep the Phase 3 comparison *controlled* (architecture is the only varying factor), this notebook re-trains ResNet-18 under the unified recipe above. The original Phase 2 result remains the project's "first baseline" and is referenced in the final report.


## 1 — Setup

In [ ]:
# Run this cell once per Colab session.
# If torch/torchvision are pre-installed (Colab default), pip will be a no-op.
import sys, subprocess
def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Ensure scikit-learn and pandas are present (Colab has them by default; safe to skip).
try:
    import sklearn, pandas  # noqa: F401
except ImportError:
    pip_install(['scikit-learn', 'pandas'])

import os, json, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# cudnn.benchmark=True trades exact determinism for speed; acceptable here since we
# report mean over runs in the thesis. Set deterministic=True if exact reproducibility
# is required (slower).
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
# Optional: mount Google Drive for persistent checkpoints/results
USE_DRIVE = True  # set False if not running in Colab
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        ROOT = '/content/drive/MyDrive/thesis/phase3'
    except Exception as e:
        print('Drive not available, using local path. Reason:', e)
        ROOT = '/content/phase3'
else:
    ROOT = './phase3'
os.makedirs(ROOT, exist_ok=True)
print('Output root:', ROOT)


## 2 — Configuration

Centralised config so every model sees the same recipe (only the architecture differs).

> **Compute budget note.** Default `epochs=50` is chosen so that all three models can be trained in one Colab session (~1.5–2 h on a T4). For the final thesis results, increase to 100–200 epochs and run with multiple seeds (3 is standard for the thesis).


In [ ]:
CFG = dict(
    img_size=32,
    num_classes=10,
    batch_size=128,
    epochs=30,
    warmup_epochs=3,
    lr=5e-4,
    weight_decay=0.05,
    label_smoothing=0.1,
    mixup_alpha=0.8,
    cutmix_alpha=1.0,
    mix_prob=0.5,          # probability of applying Mixup/CutMix per batch
    randaug_n=2,
    randaug_m=9,
    grad_clip=1.0,
    use_amp=True,
    data_fraction=1.0,     # 1.0 = full data; 0.1 = 10% regime (thesis low-data setting)
    val_fraction=0.10,     # 10 % of training data held out for validation
    num_workers=2,
    seed=SEED,
)
print(json.dumps(CFG, indent=2))


## 3 — Data loading and preparation

The same stratified 90/10 train/val split from Phase 2 is reused. The held-out validation set drives early stopping (best-val-acc model is restored before test evaluation). The test set is *never* used during training.


In [ ]:
# Channel statistics computed on the CIFAR-10 training set (standard published values).
# Computing them on the training split only avoids data leakage into the test set.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
    T.RandAugment(num_ops=CFG['randaug_n'], magnitude=CFG['randaug_m']),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    T.RandomErasing(p=0.25),
])
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

DATA_ROOT = '/content/data'
train_full = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True,
                                          transform=train_transform)
val_basis  = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True,
                                          transform=eval_transform)  # clean transforms for val
test_set   = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True,
                                          transform=eval_transform)
CLASS_NAMES = train_full.classes
print('Classes:', CLASS_NAMES)


In [ ]:
def stratified_split_indices(targets, val_fraction, seed):
    '''Stratified 1 - val_fraction / val_fraction split. Returns (train_idx, val_idx).'''
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    train_idx, val_idx = [], []
    for c in np.unique(targets):
        idxs = np.where(targets == c)[0]
        rng.shuffle(idxs)
        n_val = int(round(len(idxs) * val_fraction))
        val_idx.extend(idxs[:n_val].tolist())
        train_idx.extend(idxs[n_val:].tolist())
    rng.shuffle(train_idx); rng.shuffle(val_idx)
    return train_idx, val_idx

def stratified_subset(indices, targets, fraction, seed):
    '''Class-stratified subsample of `indices` keeping `fraction` of each class.
    Used to construct low-data regimes (10 %, 5 %, 1 %) consistently across models.'''
    if fraction >= 1.0:
        return list(indices)
    rng = np.random.default_rng(seed)
    indices = np.asarray(indices)
    cls_of  = np.asarray(targets)[indices]
    keep = []
    for c in np.unique(cls_of):
        cls_idx = indices[cls_of == c]
        rng.shuffle(cls_idx)
        n = int(round(len(cls_idx) * fraction))
        keep.extend(cls_idx[:n].tolist())
    rng.shuffle(keep)
    return keep

# ---- Updated usage ----
all_targets = train_full.targets

# train_idx_base = the full 90% pool. Per-regime subsampling happens
# inside the training loop so all models see the same indices at each regime.
train_idx_base, val_idx = stratified_split_indices(
    all_targets, CFG['val_fraction'], CFG['seed']
)
val_subset = Subset(val_basis, val_idx)  # clean (eval) transforms

print(f'Full train pool : {len(train_idx_base):>5d} images')
print(f'Val (fixed)     : {len(val_idx):>5d} images')
print(f'Test (fixed)    : {len(test_set):>5d} images')

# val and test loaders are fixed across all regimes
val_loader  = DataLoader(val_subset, batch_size=256, shuffle=False,
                         num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_set,   batch_size=256, shuffle=False,
                         num_workers=CFG['num_workers'], pin_memory=True)
CLASS_NAMES = train_full.classes
print('Classes:', CLASS_NAMES)

## 4 — Mixup / CutMix

Implemented manually (rather than via `timm`) for transparency in the thesis. Each batch has probability `mix_prob` of being mixed; conditional on mixing, Mixup and CutMix are equally likely.


In [ ]:
def rand_bbox(size, lam):
    '''Random bounding box for CutMix patch.'''
    W, H = size[2], size[3]
    cut_rat = math.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W); bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W); bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def apply_mixup_cutmix(x, y, cfg):
    '''Returns (mixed_x, (y_a, y_b, lam)). lam=1.0 => no mixing.'''
    if np.random.rand() > cfg['mix_prob']:
        return x, (y, y, 1.0)
    if np.random.rand() < 0.5:
        # CutMix
        lam = np.random.beta(cfg['cutmix_alpha'], cfg['cutmix_alpha'])
        idx = torch.randperm(x.size(0), device=x.device)
        bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
        x[:, :, bbx1:bbx2, bby1:bby2] = x[idx, :, bbx1:bbx2, bby1:bby2]
        lam = 1.0 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
        return x, (y, y[idx], lam)
    else:
        # Mixup
        lam = np.random.beta(cfg['mixup_alpha'], cfg['mixup_alpha'])
        idx = torch.randperm(x.size(0), device=x.device)
        x = lam * x + (1 - lam) * x[idx]
        return x, (y, y[idx], lam)

def mixed_loss(criterion, logits, targets_tuple):
    y_a, y_b, lam = targets_tuple
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


## 5 — Baseline reference: ResNet-18 (CNN)

Same CIFAR-adapted stem as in Phase 2: a single 3×3 conv stride 1, replacing the ImageNet 7×7+maxpool stem that would otherwise collapse 32×32 → 8×8 before any residual block runs.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet18Cifar(nn.Module):
    '''ResNet-18 with CIFAR stem (Phase 2 architecture).'''
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer( 64,  64, 2, stride=1)
        self.layer2 = self._make_layer( 64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(512, num_classes)
    def _make_layer(self, in_planes, planes, n, stride):
        layers = [BasicBlock(in_planes, planes, stride)]
        for _ in range(1, n):
            layers.append(BasicBlock(planes, planes, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.pool(x).flatten(1))


## 6 — Shared transformer building blocks

Both the ViT (Model 1) and the CCT (Model 2) use the same encoder block. Isolating these primitives keeps the comparison clean: the only difference between the two models is **how tokens are produced** (patch projection vs convolutional tokenizer) and **how the final representation is pooled** (`[CLS]` token vs learnable sequence pooling).


In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

class Attention(nn.Module):
    def __init__(self, dim, num_heads, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))

class DropPath(nn.Module):
    '''Stochastic depth (Huang et al., 2016) at the sample level inside residual branches.'''
    def __init__(self, p=0.0):
        super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep).div_(keep)
        return x * mask

class Block(nn.Module):
    '''Pre-norm transformer encoder block (standard ViT/DeiT design).'''
    def __init__(self, dim, num_heads, mlp_ratio=4.0,
                 drop=0.0, attn_drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = Attention(dim, num_heads, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, int(dim * mlp_ratio), drop=drop)
    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

def init_transformer_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)


## 7 — Part A · Advanced Model 1: ViT-Small/4

**Architecture.** Splits the 32×32 input into 64 non-overlapping 4×4 patches via a strided convolution, prepends a learnable `[CLS]` token, adds learnable 1-D positional embeddings, and runs 12 standard pre-norm transformer blocks (embed dim 384, 6 heads, MLP ratio 4). Classification uses only the `[CLS]` representation after a final LayerNorm.

**Why suitable.** Directly answers the thesis question: is a model with *no* convolutional inductive bias competitive on a small dataset trained from scratch? Per Dosovitskiy et al. (2021) and Steiner et al. (2022), the expected outcome is that ViT underperforms a similarly-sized CNN at this scale, with the gap closeable but not eliminable by heavy augmentation.

**Differences from baseline.**

| Property | ResNet-18 | ViT-Small/4 |
|---|---|---|
| Inductive biases | Locality, translation equivariance, hierarchy | None beyond positional embeddings |
| Receptive field | Grows with depth | Global from layer 1 |
| Normalisation | BatchNorm | LayerNorm |
| Activations | ReLU | GELU |
| Token count | n/a (CNN feature maps) | 65 (64 patches + CLS) |
| Parameters | ~11 M | ~22 M |


In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)              # B, C, H/P, W/P
        return x.flatten(2).transpose(1, 2)  # B, N, C

class ViTSmall4(nn.Module):
    '''ViT-Small with 4x4 patches. 64 patch tokens + 1 CLS token on CIFAR-10.'''
    def __init__(self, img_size=32, patch_size=4, in_chans=3, num_classes=10,
                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0,
                 drop=0.0, attn_drop=0.0, drop_path=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        n_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(drop)
        dpr = [x.item() for x in torch.linspace(0, drop_path, depth)]
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, drop, attn_drop, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(init_transformer_weights)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])


## 8 — Part B · Advanced Model 2: CCT-7/3×1

**Architecture (Hassani et al., 2022).** Two design changes vs ViT:

1. **Convolutional tokenizer** — a 3×3 convolution + ReLU + 3×3 max-pool replaces patch splitting. This reintroduces locality and hierarchical downsampling into the front-end while preserving the transformer encoder.
2. **Sequence pooling** — replaces the `[CLS]` token with a learnable attention pool over all tokens (`SeqPool`). This makes every token contribute to the classification decision, weighted by learned attention scores.

The transformer encoder is then 7 blocks deep, with embed dim 256 and 4 heads — substantially smaller than ViT-Small (~3.7 M params vs ~22 M).

**Why suitable.** CCT is the canonical reference architecture in the "transformers on small datasets" literature. It directly tests whether the thesis hypothesis — that combining convolutional priors with transformer flexibility yields better data efficiency — holds in practice.

**Differences from Model 1 (ViT).**

| Property | ViT-Small/4 | CCT-7/3×1 |
|---|---|---|
| Tokenizer | Linear patch projection | 3×3 conv + ReLU + max-pool (locality-aware) |
| Class representation | Learnable `[CLS]` token | Learnable sequence pooling |
| Depth | 12 transformer blocks | 7 transformer blocks |
| Embed dim | 384 | 256 |
| Parameters | ~22 M | ~3.7 M |


In [ ]:
class ConvTokenizer(nn.Module):
    '''3x3 conv + ReLU + maxpool. The `1` in CCT-7/3x1 = 1 conv block.'''
    def __init__(self, in_chans=3, embed_dim=256, kernel_size=3, n_conv_layers=1):
        super().__init__()
        layers = []
        c_in = in_chans
        for i in range(n_conv_layers):
            c_out = embed_dim if i == n_conv_layers - 1 else embed_dim // 2
            layers += [
                nn.Conv2d(c_in, c_out, kernel_size=kernel_size, stride=1,
                          padding=kernel_size // 2, bias=False),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            ]
            c_in = c_out
        self.tokenizer = nn.Sequential(*layers)
    def forward(self, x):
        x = self.tokenizer(x)               # B, C, H', W'
        return x.flatten(2).transpose(1, 2)  # B, N, C

class SeqPool(nn.Module):
    '''Learnable attention pool over the token sequence (replaces [CLS] token).'''
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)
    def forward(self, x):                          # B, N, C
        w = self.attn(x).softmax(dim=1)             # B, N, 1
        return (w.transpose(1, 2) @ x).squeeze(1)  # B, C

class CCT_7_3x1(nn.Module):
    '''Compact Convolutional Transformer (CCT-7/3x1) — Hassani et al. 2022.
    7 transformer blocks, 3x3 conv tokenizer with 1 conv block. ~3.7M params.'''
    def __init__(self, img_size=32, in_chans=3, num_classes=10,
                 embed_dim=256, depth=7, num_heads=4, mlp_ratio=2.0,
                 drop=0.1, attn_drop=0.1, drop_path=0.1):
        super().__init__()
        self.tokenizer = ConvTokenizer(in_chans, embed_dim,
                                       kernel_size=3, n_conv_layers=1)
        # Determine token count from a dry-run on a dummy input
        with torch.no_grad():
            n_tokens = self.tokenizer(
                torch.zeros(1, in_chans, img_size, img_size)
            ).shape[1]
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
        self.pos_drop  = nn.Dropout(drop)
        dpr = [x.item() for x in torch.linspace(0, drop_path, depth)]
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, drop, attn_drop, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.pool = SeqPool(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(init_transformer_weights)
    def forward(self, x):
        x = self.tokenizer(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(self.pool(x))


## 9 — Sanity check: parameter counts and forward pass

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy = torch.zeros(2, 3, 32, 32)
for ctor, name in [(ResNet18Cifar, 'ResNet-18'),
                   (ViTSmall4,     'ViT-Small/4'),
                   (CCT_7_3x1,     'CCT-7/3x1')]:
    m = ctor()
    out = m(dummy)
    assert out.shape == (2, 10), f'{name} output shape wrong: {out.shape}'
    print(f'{name:15s} | params: {count_params(m)/1e6:>6.2f}M | output: {tuple(out.shape)}')
    del m


## 10 — Training pipeline

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

def cosine_warmup(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    '''Linear warmup then cosine decay to 0. Called per step.'''
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_pred, all_y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total   += x.size(0)
        all_pred.append(pred.cpu().numpy()); all_y.append(y.cpu().numpy())
    return loss_sum / total, correct / total, np.concatenate(all_y), np.concatenate(all_pred)


In [ ]:
def train_model(model, save_key, train_loader, val_loader, test_loader, cfg):
    '''Train `model` under `cfg`. Saves best-val checkpoint and a results JSON.

    Returns a dict with metrics, history, and test-set predictions.
    '''
    model = model.to(DEVICE)
    criterion      = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    eval_criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
    scheduler = cosine_warmup(optimizer, cfg['warmup_epochs'],
                              cfg['epochs'], len(train_loader))
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['use_amp'] and DEVICE == 'cuda')

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  [], 'lr': []}
    best = {'val_acc': 0.0, 'state': None, 'epoch': -1}
    t_start = time.time()

    for epoch in range(cfg['epochs']):
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            x, y_mix = apply_mixup_cutmix(x, y, cfg)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=cfg['use_amp'] and DEVICE == 'cuda'):
                logits = model(x)
                loss = mixed_loss(criterion, logits, y_mix)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            with torch.no_grad():
                loss_sum += loss.item() * x.size(0)
                y_a, y_b, lam = y_mix
                pred = logits.argmax(1)
                # Training "accuracy" against the dominant target — for monitoring only.
                correct += (lam * (pred == y_a).float()
                            + (1 - lam) * (pred == y_b).float()).sum().item()
                total   += x.size(0)
        train_loss = loss_sum / total; train_acc = correct / total
        val_loss, val_acc, _, _ = evaluate(model, val_loader, eval_criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        if val_acc > best['val_acc']:
            best = {'val_acc': val_acc,
                    'state':  {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    'epoch':  epoch}

        print(f'[{save_key}] ep {epoch+1:3d}/{cfg["epochs"]} | '
              f'train {train_loss:.3f}/{train_acc*100:.2f}% | '
              f'val {val_loss:.3f}/{val_acc*100:.2f}% | '
              f'lr {optimizer.param_groups[0]["lr"]:.2e}')

    train_time = time.time() - t_start

    # Restore best-val weights, then evaluate on the held-out test set
    model.load_state_dict(best['state'])
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, eval_criterion)

    # Inference timing (per image, batched at 64)
    model.eval()
    with torch.no_grad():
        x_dummy = torch.randn(64, 3, 32, 32, device=DEVICE)
        for _ in range(3):
            _ = model(x_dummy)               # warmup
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
        for _ in range(20):
            _ = model(x_dummy)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        infer_ms = (time.time() - t1) / 20 / 64 * 1000

    result = {
        'name': save_key,
        'params_M': count_params(model) / 1e6,
        'epochs': cfg['epochs'],
        'best_val_acc': best['val_acc'],
        'best_epoch': best['epoch'],
        'test_acc': test_acc,
        'test_loss': test_loss,
        'train_time_min': train_time / 60,
        'inference_ms_per_image': infer_ms,
        'history': history,
        'y_true': y_true.tolist(),
        'y_pred': y_pred.tolist(),
        'train_acc_final': history['train_acc'][-1],
        'val_acc_final':   history['val_acc'][-1],
        'overfit_gap_final': history['train_acc'][-1] - history['val_acc'][-1],
    }
    # Persist
    safe_key = save_key.replace('/', '_')
    with open(os.path.join(ROOT, f'{safe_key}_results.json'), 'w') as f:
        json.dump(result, f, indent=2, default=float)
    torch.save(best['state'], os.path.join(ROOT, f'{safe_key}_best.pt'))
    print(f'[{save_key}] DONE | test acc {test_acc*100:.2f}% | best val {best["val_acc"]*100:.2f}% '
          f'(epoch {best["epoch"]+1}) | train time {train_time/60:.1f} min')
    return result


## 11 — Train all three models

> Expect ~1.5–2 hours on a Colab T4 GPU with `epochs=50`. Each model writes its checkpoint and JSON to the Drive folder, so the cell is safe to re-run for a single model by commenting out the others.


In [ ]:
# Regimes to compare. For the full thesis grid add 0.50 and 0.05.
# 100% / 10% / 1% is the minimum meaningful set for the Phase 3 deliverable.
REGIMES = [1.0, 0.10, 0.01]

FAMILY = {'ResNet-18': 'CNN', 'ViT-Small/4': 'ViT', 'CCT-7/3x1': 'Hybrid'}
MODEL_CTORS = [
    ('ResNet-18',   ResNet18Cifar),
    ('ViT-Small/4', ViTSmall4),
    ('CCT-7/3x1',   CCT_7_3x1),
]

# Nested dict: RESULTS[model_name][fraction] = result_dict
RESULTS = {name: {} for name, _ in MODEL_CTORS}

for fraction in REGIMES:
    pct_label = f'{int(fraction * 100):03d}pct'
    regime_idx = stratified_subset(train_idx_base, all_targets, fraction, CFG['seed'])
    regime_loader = DataLoader(
        Subset(train_full, regime_idx),
        batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
    )
    n_train = len(regime_idx)
    print(f'\n{"="*60}')
    print(f'REGIME {fraction*100:.0f}%  —  {n_train} training images')
    print(f'{"="*60}')

    for model_name, ctor in MODEL_CTORS:
        save_key = f'{model_name.replace("/", "_")}_{pct_label}'
        print(f'\n--- {model_name} @ {fraction*100:.0f}% ---')
        result = train_model(
            ctor(num_classes=CFG['num_classes']),
            save_key, regime_loader, val_loader, test_loader, CFG,
        )
        result['fraction']   = fraction
        result['model_name'] = model_name
        result['n_train']    = n_train
        RESULTS[model_name][fraction] = result

print('\nAll training runs complete.')

## 12 — Part C · Final Model Comparison

### 12.1 Comparison table (Part C requirement 1)


In [ ]:
import pandas as pd

# ---- Flat table (one row per model × regime) ----
rows = []
for model_name, regime_dict in RESULTS.items():
    for fraction, r in sorted(regime_dict.items(), reverse=True):
        rows.append({
            'Model':              model_name,
            'Family':             FAMILY[model_name],
            'Regime':             f'{int(fraction*100)}%',
            'N train':            r['n_train'],
            'Params (M)':         round(r['params_M'], 2),
            'Best Val Acc (%)':   round(r['best_val_acc']  * 100, 2),
            'Test Acc (%)':       round(r['test_acc']       * 100, 2),
            'Overfit Gap (pp)':   round(r['overfit_gap_final'] * 100, 2),
            'Best Epoch':         r['best_epoch'] + 1,
            'Train Time (min)':   round(r['train_time_min'], 1),
        })
df_flat = pd.DataFrame(rows)

# ---- Pivot table: architecture vs data regime ----
df_pivot = df_flat.pivot_table(
    index=['Model', 'Family'],
    columns='Regime',
    values='Test Acc (%)',
)
# Sort columns from most to least data
regime_order = [f'{int(f*100)}%' for f in sorted(REGIMES, reverse=True)]
df_pivot = df_pivot[[c for c in regime_order if c in df_pivot.columns]]
df_pivot['Params (M)'] = [RESULTS[m][REGIMES[0]]['params_M'] for m in df_pivot.index.get_level_values('Model')]

print('\n=== TEST ACCURACY BY REGIME (%) ===')
print(df_pivot.to_string())

print('\n=== FULL RESULTS TABLE ===')
print(df_flat.to_string(index=False))

df_flat.to_csv(os.path.join(ROOT, 'comparison_flat.csv'),  index=False)
df_pivot.to_csv(os.path.join(ROOT, 'comparison_pivot.csv'))

### 12.2 Overfitting analysis — learning curves (Part C requirement 2)

Train and validation curves on the same axes reveal:
- **Convergence speed** — which architecture reaches a given accuracy first.
- **Train–val gap** — direct evidence of overfitting (larger gap = more memorisation).
- **Stability** — oscillation amplitude in late epochs.


In [ ]:
colors = {'ResNet-18': 'tab:blue', 'ViT-Small/4': 'tab:orange', 'CCT-7/3x1': 'tab:green'}
markers = {'ResNet-18': 'o', 'ViT-Small/4': 's', 'CCT-7/3x1': '^'}
fractions_sorted = sorted(REGIMES)
x_labels = [f'{int(f*100)}%' for f in fractions_sorted]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Left: degradation curve (THE key thesis visual) ----
ax = axes[0]
for model_name in RESULTS:
    y = [RESULTS[model_name][f]['test_acc'] * 100 for f in fractions_sorted]
    ax.plot(fractions_sorted, y,
            color=colors[model_name], marker=markers[model_name],
            linewidth=2, markersize=7, label=model_name)
    for f, acc in zip(fractions_sorted, y):
        ax.annotate(f'{acc:.1f}', (f, acc),
                    textcoords='offset points', xytext=(4, 4), fontsize=8,
                    color=colors[model_name])
ax.set_xscale('log')
ax.set_xticks(fractions_sorted)
ax.set_xticklabels(x_labels)
ax.set_xlabel('Training data fraction (log scale)')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('Architecture degradation under data scarcity')
ax.legend(); ax.grid(alpha=0.3)

# ---- Right: overfitting gap per regime ----
ax2 = axes[1]
x = np.arange(len(fractions_sorted))
width = 0.25
for i, (model_name, offset) in enumerate(zip(RESULTS, [-width, 0, width])):
    gaps = [RESULTS[model_name][f]['overfit_gap_final'] * 100 for f in fractions_sorted]
    bars = ax2.bar(x + offset, gaps, width=width,
                   color=colors[model_name], alpha=0.8, label=model_name)
ax2.set_xticks(x); ax2.set_xticklabels(x_labels)
ax2.set_xlabel('Data regime')
ax2.set_ylabel('Train acc − Val acc (pp)')
ax2.set_title('Overfitting gap per regime\n(larger = more memorisation)')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'degradation_and_overfit.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show per-epoch curves for the most interesting regime (10 % by default).
# Change SHOW_REGIME to inspect another regime.
SHOW_REGIME = 0.10

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for model_name in RESULTS:
    r = RESULTS[model_name][SHOW_REGIME]
    c = colors[model_name]
    axes[0].plot(r['history']['train_loss'], color=c, linestyle='--', alpha=0.7,
                 label=f'{model_name} train')
    axes[0].plot(r['history']['val_loss'],   color=c, linestyle='-',
                 label=f'{model_name} val')
    axes[1].plot(np.array(r['history']['train_acc']) * 100, color=c, linestyle='--', alpha=0.7,
                 label=f'{model_name} train')
    axes[1].plot(np.array(r['history']['val_acc'])   * 100, color=c, linestyle='-',
                 label=f'{model_name} val')

for ax, ylabel, title in zip(axes,
                             ['Loss', 'Accuracy (%)'],
                             [f'Loss curves @ {int(SHOW_REGIME*100)}% data',
                              f'Accuracy curves @ {int(SHOW_REGIME*100)}% data']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, f'learning_curves_{int(SHOW_REGIME*100)}pct.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Numeric overfitting summary across all regimes
print(f'\n{"Model":15s} | {"Regime":>7s} | {"Train":>8s} | {"Val":>8s} | {"Test":>8s} | {"T-V gap":>8s}')
print('-' * 72)
for model_name in RESULTS:
    for f in sorted(RESULTS[model_name], reverse=True):
        r = RESULTS[model_name][f]
        gap = (r['train_acc_final'] - r['val_acc_final']) * 100
        print(f'{model_name:15s} | {int(f*100):>6d}% | '
              f'{r["train_acc_final"]*100:7.2f}% | {r["val_acc_final"]*100:7.2f}% | '
              f'{r["test_acc"]*100:7.2f}% | {gap:7.2f}')

### 12.3 Confusion matrices on the held-out test set

In [ ]:
SHOW_REGIMES_CM = [1.0, 0.01]  # full data ceiling and extreme low-data
n_regimes = len(SHOW_REGIMES_CM)
n_models  = len(RESULTS)

fig, axes = plt.subplots(n_regimes, n_models,
                         figsize=(5.5 * n_models, 5 * n_regimes))
for row_i, fraction in enumerate(SHOW_REGIMES_CM):
    for col_j, model_name in enumerate(RESULTS):
        ax = axes[row_i][col_j]
        r  = RESULTS[model_name][fraction]
        cm = confusion_matrix(r['y_true'], r['y_pred'])
        ax.imshow(cm, cmap='Blues')
        ax.set_title(f'{model_name} @ {int(fraction*100)}%\n'
                     f'Test acc: {r["test_acc"]*100:.2f}%', fontsize=9)
        ax.set_xticks(range(10)); ax.set_yticks(range(10))
        ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(CLASS_NAMES, fontsize=7)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        thresh = cm.max() / 2.0
        for i in range(10):
            for j in range(10):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=6,
                        color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

### 12.4 Per-class precision / recall / F1

In [ ]:
# Report per-class metrics at the most informative regime for the thesis (10%)
REPORT_REGIME = 0.10
for model_name in RESULTS:
    r = RESULTS[model_name][REPORT_REGIME]
    print(f'\n=== {model_name} @ {int(REPORT_REGIME*100)}% data ===')
    print(classification_report(r['y_true'], r['y_pred'],
                                target_names=CLASS_NAMES, digits=4))

### 12.5 Paired sample-level comparison

McNemar-style contingency on the test set. For each pair of models, count test samples where:
- Both are correct
- Only model A is correct (A wins)
- Only model B is correct (B wins)
- Both are wrong

A larger "A wins"–"B wins" asymmetry indicates the models make systematically different errors, not just noise.


In [ ]:
# Paired comparison at each regime separately
for fraction in sorted(REGIMES, reverse=True):
    names = list(RESULTS.keys())
    print(f'\n--- Regime {int(fraction*100)}% ---')
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = np.array(RESULTS[names[i]][fraction]['y_pred'])
            b = np.array(RESULTS[names[j]][fraction]['y_pred'])
            y = np.array(RESULTS[names[i]][fraction]['y_true'])
            both    = int(((a == y) & (b == y)).sum())
            a_only  = int(((a == y) & (b != y)).sum())
            b_only  = int(((b == y) & (a != y)).sum())
            neither = int(((a != y) & (b != y)).sum())
            n_dis   = a_only + b_only
            chi2    = (abs(a_only - b_only) - 1) ** 2 / n_dis if n_dis > 0 else 0.0
            print(f'  {names[i]:15s} vs {names[j]:15s} | '
                  f'both {both:4d} | only A {a_only:4d} | only B {b_only:4d} | '
                  f'neither {neither:4d} | McNemar chi²={chi2:.2f}')

### 12.6 Model selection justification (Part C requirement 3)

Selection is based on **test accuracy** as the primary metric. Parameter count and inference latency are reported alongside to inform efficiency-aware deployment decisions, but they are **not** used to override the accuracy ranking — Phase 3 explicitly requires selection on metrics, not preference.


In [ ]:
print('=== MODEL SELECTION BY REGIME ===\n')
for fraction in sorted(REGIMES, reverse=True):
    candidates = [(m, RESULTS[m][fraction]) for m in RESULTS]
    ranked = sorted(candidates, key=lambda kv: kv[1]['test_acc'], reverse=True)
    winner, wr = ranked[0]
    print(f'Regime {int(fraction*100):>3d}%  ({wr["n_train"]:>5d} images):')
    for rank, (m, r) in enumerate(ranked, 1):
        marker = ' <-- best' if rank == 1 else ''
        print(f'  {rank}. {m:15s} test {r["test_acc"]*100:.2f}%  '
              f'overfit gap {r["overfit_gap_final"]*100:.2f} pp{marker}')
    print()

# Summary table: which architecture wins at each regime
print('=== WINNER PER REGIME ===')
for fraction in sorted(REGIMES, reverse=True):
    best_m = max(RESULTS, key=lambda m: RESULTS[m][fraction]['test_acc'])
    best_acc = RESULTS[best_m][fraction]['test_acc'] * 100
    print(f'  {int(fraction*100):>3d}%  →  {best_m}  ({best_acc:.2f}%)')

print('\nKey thesis interpretation:')
print('  If the same model wins at all regimes: architecture dominance is consistent.')
print('  If the winner changes: there is a crossover point — the central finding of the thesis.')
print('  Cross-reference with the overfit gap to distinguish genuine generalisation from luck.')

## 13 — Persist consolidated artefacts

In [ ]:
final_payload = {
    'config':  CFG,
    'regimes': REGIMES,
    'results': {
        model_name: {
            str(f): {k: v for k, v in r.items()
                     if k not in ('history', 'y_true', 'y_pred')}
            for f, r in regime_dict.items()
        }
        for model_name, regime_dict in RESULTS.items()
    },
    'pivot_table': df_pivot.reset_index().to_dict(orient='records'),
}
with open(os.path.join(ROOT, 'phase3_final_results.json'), 'w') as f:
    json.dump(final_payload, f, indent=2, default=float)

print('Saved artefacts to:', ROOT)
for fname in sorted(os.listdir(ROOT)):
    size_kb = os.path.getsize(os.path.join(ROOT, fname)) / 1024
    print(f'  {fname:<45s} {size_kb:6.1f} KB')

## 15 — Notes for the Phase 3 report

The following deliverable checklist maps the artefacts above onto the Phase 3 specification:

| Spec requirement | Artefact / location |
|---|---|
| Part A · Model 1 description | Section 7 (markdown) |
| Part A · Training setup | Section 3 (CFG) + Section 10 (`train_model`) |
| Part A · Results vs baseline | Section 12.1 (table), 12.4 (per-class) |
| Part B · Model 2 description | Section 8 (markdown) |
| Part B · Comparison with baseline and Model 1 | Sections 12.1 – 12.5 |
| Part C · Comparison table | Section 12.1 → `comparison_table.csv` |
| Part C · Overfitting analysis | Section 12.2 → `learning_curves.png`, `overfit_gap.png` |
| Part C · Model selection (metric-based) | Section 12.6 |

**Limitations to acknowledge in the written report (so the supervisor sees they are understood, not overlooked):**

1. **Single seed.** Phase 3 reports results from one seed. The thesis plan calls for 3 seeds per configuration; this will be added in the next phase along with paired t-tests / Wilcoxon signed-rank tests.
2. **Single data regime.** Only the 100 % regime is run here. The full thesis comparison requires the 50 % / 10 % / 5 % / 1 % regimes (Section 14 provides the scaffold).
3. **Epoch budget.** 50 epochs is below typical converged-from-scratch budgets for transformers on CIFAR-10 (literature reports 200–300 epochs for CCT). Final thesis numbers should be re-run with 100–200 epochs.
4. **No knowledge distillation / no pre-training.** Intentional: isolates architectural inductive biases as the only varying factor.
